# Model 2 - VGG Sequential
Notebook train và test VGG Sequential.

In [ ]:
# 1. Import và cấu hình
import sys
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report

sys.path.append('../py')
from metrics import calculate_metrics

DATA_DIR = Path('../../../data/VN_trash_classification_preprocessing')
CHECKPOINT_DIR = Path('../checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 30
PATIENCE = 15
BATCH_SIZE = 32
LR = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2. Xử lý ảnh và load dữ liệu
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

train_ds = ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_ds = ImageFolder(DATA_DIR / 'val', transform=test_transform)
test_ds = ImageFolder(DATA_DIR / 'test', transform=test_transform)

# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {'num_workers': 8, 'pin_memory': True, 'persistent_workers': True}
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_options)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, **loader_options)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, **loader_options)

class_names = train_ds.classes
print('Device:', device, '| Classes:', class_names)
print(len(train_ds), len(val_ds), len(test_ds))

In [ ]:
# 3. Tạo VGG Sequential
class VGGSequential(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        # 5 stage VGG16: số conv mỗi stage là 2/2/3/3/3, sau mỗi stage có MaxPool
        layers = []
        in_channels = 3
        for width, count in ((64, 2), (128, 2), (256, 3), (512, 3), (512, 3)):
            for _ in range(count):
                layers += [
                    nn.Conv2d(in_channels, width, kernel_size=3, padding=1),
                    nn.BatchNorm2d(width),
                    nn.ReLU(inplace=True),
                ]
                in_channels = width
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.6),
            nn.Linear(128, number_of_classes),
        )

    def forward(self, images):
        return self.classifier(self.features(images))


model = VGGSequential(len(class_names)).to(device)
print(model)

In [ ]:
# 4. Loss, optimizer và scheduler
counts = Counter(train_ds.targets)
weights = torch.tensor(
    [len(train_ds) / (len(class_names) * counts[i]) for i in range(len(class_names))],
    dtype=torch.float32,
    device=device,
)

loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

In [ ]:
# 5. Train và validation
def run_epoch(loader, training=False):
    model.train(training)
    total_loss = 0.0
    correct = 0
    seen = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        seen += labels.size(0)

    return total_loss / seen, correct / seen


best_acc = -1.0
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader)
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f'Epoch {epoch + 1}/{EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%}'
    )

    # Dừng khi validation accuracy không tăng trong PATIENCE epoch liên tiếp.
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping tại epoch {epoch + 1}')
        break

model.load_state_dict(best_state)
print('Best epoch:', best_epoch, '| Best validation accuracy:', f'{best_acc:.2%}')

In [ ]:
# 6. Test và đánh giá
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

metrics = calculate_metrics(true_labels, predicted_labels)

print()
print('Test Result')
print(f'Test Accuracy: {metrics["accuracy"]:.2%}')
print(f'Macro Precision: {metrics["macro_precision"]:.4f}')
print(f'Macro Recall: {metrics["macro_recall"]:.4f}')
print(f'Macro F1: {metrics["macro_f1"]:.4f}')
print(f'Weighted F1: {metrics["weighted_f1"]:.4f}')
print()
print(classification_report(true_labels, predicted_labels, target_names=class_names, digits=4, zero_division=0))

In [ ]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR / 'vgg_sequential.pt')